In [ ]:
import plotly.express as px
import plotly.graph_objects as go
import polars as pl
import polars.selectors as cs
import torch

from aare.params import read_params
from aare.utils import METRICS_FOLDER

# Draft for the model report

Final one will be clean and if possible using Marimo, this is just the playground to think about the charts etc.

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (16, 9)

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
torch.set_float32_matmul_precision("medium")

In [ ]:
params = read_params()
tz = params["general"]["timezone"]

In [ ]:
raw_metrics = pl.read_csv(METRICS_FOLDER / "raw" / "LR-dev.csv")
raw_metrics = raw_metrics.with_columns(pl.col("run_ts", "time").str.to_datetime(time_zone=tz))
raw_metrics

In [ ]:
from datetime import timedelta

raw_metrics = raw_metrics.with_columns(lag=((pl.col("time") - pl.col("run_ts")) / timedelta(hours=1) + 1).cast(int))
raw_metrics

In [ ]:
raw_metrics = raw_metrics.with_columns(ae=pl.col("err").abs(), adpd=pl.col("dpd").abs())
raw_metrics

In [ ]:
px.box(raw_metrics, x="lag", y="ae", points=False)

In [ ]:
col = pl.col("err")
df = (
    raw_metrics.lazy()
    .sort("time")
    .group_by("time")
    # .group_by_dynamic("time", every="1d")
    .agg(
        col.abs().median(),
        col.abs().quantile(0.25).name.suffix("_q25"),
        col.abs().quantile(0.75).name.suffix("_q75"),
    )
    .sort("time")
    .with_columns(cs.float().rolling_mean(24 * 30, center=True))
    .collect()
)
df

In [ ]:
fig = go.Figure(
    [
        go.Scatter(
            name="err",
            x=df["time"],
            y=df["err"],
            mode="lines",
        ),
        go.Scatter(
            name="Q75%",
            x=df["time"],
            y=df["err_q75"],
            mode="lines",
            line=dict(width=0),
            showlegend=False,
        ),
        go.Scatter(
            name="Q25%",
            x=df["time"],
            y=df["err_q25"],
            mode="lines",
            line=dict(width=0),
            showlegend=False,
            fill="tonexty",
        ),
    ]
)

fig.update_layout(
    yaxis=dict(
        title=dict(
            text="Absolute Forecast Error [°C]",
        )
    ),
    title=dict(
        text="Forecast error over validation period",
        subtitle=dict(
            text="Prediction interval shows quantiles for all different forecast runs that predicted that time (all ages)"
        ),
    ),
    hovermode="x",
)

fig

In [ ]:
raw_metrics = raw_metrics.with_columns(pl.col("time").dt.hour().alias("hour"))
raw_metrics

In [ ]:
px.violin(raw_metrics, x="hour", y="ae", points=False, box=True)

In [ ]:
raw_metrics.group_by("run_ts").agg(pl.col("ae").mean()).select(pl.col("ae").median())

In [ ]:
raw_metrics.filter(
    pl.col("time").dt.month() >= 4,
    pl.col("time").dt.month() <= 9,
    pl.col("time").dt.hour() >= 10,
    pl.col("time").dt.hour() <= 22,
).group_by("run_ts").agg(pl.col("ae").mean()).select(pl.col("ae").median())